In [1]:
from pathlib import Path
import zipfile

zip_files = list(Path("/content").glob("*.zip"))

if len(zip_files) != 1:
    print("ZIP files found:", [p.name for p in zip_files])
    raise ValueError("Upload one ZIP file, or specify its path.")

zip_path = zip_files[0]
extract_folder = Path("/content/housing_data")

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_folder)

csv_files = [
    p for p in extract_folder.rglob("*.csv")
    if "__MACOSX" not in p.parts and not p.name.startswith("._")
]

print("Extracted to:", extract_folder)
print("Number of CSV files:", len(csv_files))

Extracted to: /content/housing_data
Number of CSV files: 4


In [4]:
from pathlib import Path
import pandas as pd

matches = list(Path("/content").rglob("train_prepared_24months.csv"))
if len(matches) != 1:
    raise ValueError(f"Expected one training CSV, found: {matches}")

sales = pd.read_csv(matches[0], dtype={"PostalCode": "string"})
dates = pd.to_datetime(sales["CloseDate"], errors="raise", utc=True)

train = sales.loc[
    (dates >= "2024-09-01") & (dates < "2026-06-01")
].copy()

validation = sales.loc[
    (dates >= "2026-06-01") & (dates < "2026-09-01")
].copy()

print(f"Training sales before cutoff: {len(train):,}")
print(f"June–August validation sales: {len(validation):,}")

Training sales before cutoff: 227,747
June–August validation sales: 37,828


In [6]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

price_cutoff = 17_000_000
train_screened = train.loc[train["ClosePrice"] <= price_cutoff].copy()

numeric = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt"
]
categories = ["City", "CountyOrParish", "PostalCode"]
flags = [
    f"{column}_{suffix}"
    for column in numeric
    for suffix in ["invalid_flag", "missing_flag"]
]
features = numeric + categories + flags

X_train = train_screened[features].copy()
X_valid = validation[features].copy()

for frame in [X_train, X_valid]:
    for flag in flags:
        frame[flag] = frame[flag].astype(str).str.lower().isin(["true", "1"])
    for column in numeric:
        frame[column] = frame[column].mask(frame[f"{column}_missing_flag"])

preprocessing = ColumnTransformer([
    ("numeric", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("location", OneHotEncoder(handle_unknown="ignore"), categories),
    ("flags", "passthrough", flags),
])

model = make_pipeline(preprocessing, LinearRegression())
model.fit(X_train, train_screened["ClosePrice"])

predictions = model.predict(X_valid)
actual = validation["ClosePrice"]

print(f"Training sales excluded above $17M: {len(train) - len(train_screened):,}")
print(f"Training sales used: {len(train_screened):,}")
print(f"Validation sales scored: {len(actual):,}")
print(f"MAE:  ${mean_absolute_error(actual, predictions):,.0f}")
print(f"RMSE: ${np.sqrt(mean_squared_error(actual, predictions)):,.0f}")
print(f"R²:   {r2_score(actual, predictions):.4f}")

Training sales excluded above $17M: 245
Training sales used: 227,502
Validation sales scored: 37,828
MAE:  $307,174
RMSE: $890,219
R²:   0.6653


“I trained a linear regression model using sales of $17 million or less and evaluated it on all 37,828 sales from June through August 2026.”

In [7]:
from sklearn.base import clone

# Restore missing values using the same flags as in the screened fit.
X_train_full = train[features].copy()

for flag in flags:
    X_train_full[flag] = (
        X_train_full[flag].astype(str).str.lower().isin(["true", "1"])
    )

for column in numeric:
    X_train_full[column] = X_train_full[column].mask(
        X_train_full[f"{column}_missing_flag"]
    )

# Fit fresh preprocessing and regression on all training rows.
model_full = clone(model)
model_full.fit(X_train_full, train["ClosePrice"])
predictions_full = model_full.predict(X_valid)

comparison = pd.DataFrame([
    {
        "Model": name,
        "Training sales": count,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
    }
    for name, count, predicted in [
        ("All training sales", len(train), predictions_full),
        ("$17M training cutoff", len(train_screened), predictions),
    ]
])

display(comparison.style.format({
    "Training sales": "{:,}",
    "MAE": "${:,.0f}",
    "RMSE": "${:,.0f}",
    "R²": "{:.4f}",
}))

,Model,Training sales,MAE,RMSE,R²
0,All training sales,"227,747","$384,323","$923,397",0.6399
1,$17M training cutoff,"227,502","$307,174","$890,219",0.6653


In [8]:
from sklearn.base import clone

# Load September's prepared sales.
test_path = matches[0].with_name("test_prepared_202609.csv")
september = pd.read_csv(test_path, dtype={"PostalCode": "string"})

# Refit using both the earlier training months and the validation months.
refit_sales = pd.concat([train, validation], ignore_index=True)
final_train = refit_sales.loc[
    refit_sales["ClosePrice"] <= price_cutoff
].copy()

X_final_train = final_train[features].copy()
X_september = september[features].copy()

for frame in [X_final_train, X_september]:
    for flag in flags:
        frame[flag] = (
            frame[flag].astype(str).str.strip().str.lower().isin(["true", "1"])
        )
    for column in numeric:
        frame[column] = frame[column].mask(frame[f"{column}_missing_flag"])

final_linear = clone(model)
final_linear.fit(X_final_train, final_train["ClosePrice"])

september_actual = september["ClosePrice"]
september_pred = final_linear.predict(X_september)

print(f"Training sales excluded above $17M: {len(refit_sales) - len(final_train):,}")
print(f"Training sales used: {len(final_train):,}")
print(f"September sales scored: {len(september_actual):,}")
print(f"September MAE:  ${mean_absolute_error(september_actual, september_pred):,.0f}")
print(f"September RMSE: ${np.sqrt(mean_squared_error(september_actual, september_pred)):,.0f}")
print(f"September R²:   {r2_score(september_actual, september_pred):.4f}")
print(f"Negative September predictions: {(september_pred < 0).sum():,}")

Training sales excluded above $17M: 292
Training sales used: 265,283
September sales scored: 11,186
September MAE:  $308,208
September RMSE: $768,355
September R²:   0.7081
Negative September predictions: 103


“After validating the $17 million training cutoff on June–August 2026 sales, I refit Linear Regression on 265,283 sales from September 2024 through August 2026. I evaluated the refitted model on all 11,186 September 2026 sales.”

The biggest improvement was in MAE. On the same September homes, the average absolute error fell by about $72,753 per sale (19.1%). R² rose by 3.20 percentage points, and negative predictions fell by about half. The earlier June–August comparison also favored the cutoff, so the September results are consistent with what you saw in validation.

In [9]:
import numpy as np
import pandas as pd
from time import perf_counter
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Reuse the preprocessing fitted on the final training sales.
preprocessor = final_linear.steps[0][1]
X_train_models = preprocessor.transform(X_final_train)
X_test_models = preprocessor.transform(X_september)
y_train_models = final_train["ClosePrice"]

models = {
    "Decision Tree": DecisionTreeRegressor(
        max_depth=12,
        min_samples_leaf=50,
        random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=50,
        max_depth=16,
        min_samples_leaf=10,
        max_features=0.5,
        random_state=42,
        n_jobs=2
    )
}

train_predictions = {
    "Linear Regression": final_linear.predict(X_final_train)
}
test_predictions = {
    "Linear Regression": september_pred
}
fit_seconds = {"Linear Regression": np.nan}

for name, estimator in models.items():
    print(f"Training {name}...", flush=True)
    start = perf_counter()
    estimator.fit(X_train_models, y_train_models)
    fit_seconds[name] = perf_counter() - start

    train_predictions[name] = estimator.predict(X_train_models)
    test_predictions[name] = estimator.predict(X_test_models)
    print(f"Finished in {fit_seconds[name]:.1f} seconds.", flush=True)

baseline_r2 = r2_score(
    september_actual, test_predictions["Linear Regression"]
)

results = pd.DataFrame([
    {
        "Model": name,
        "Train R²": r2_score(y_train_models, train_predictions[name]),
        "September R²": r2_score(september_actual, predicted),
        "R² change vs baseline": r2_score(september_actual, predicted) - baseline_r2,
        "September MAE": mean_absolute_error(september_actual, predicted),
        "September RMSE": np.sqrt(
            mean_squared_error(september_actual, predicted)
        ),
        "Negative predictions": int((predicted < 0).sum()),
        "Fit seconds": fit_seconds[name]
    }
    for name, predicted in test_predictions.items()
])

print(f"Training sales: {len(y_train_models):,}")
print(f"September sales scored: {len(september_actual):,}")

display(results.style.format({
    "Train R²": "{:.4f}",
    "September R²": "{:.4f}",
    "R² change vs baseline": "{:+.4f}",
    "September MAE": "${:,.0f}",
    "September RMSE": "${:,.0f}",
    "Negative predictions": "{:,}",
    "Fit seconds": "{:.1f}"
}, na_rep="—"))


Training Decision Tree...
Finished in 3.8 seconds.
Training Random Forest...
Finished in 112.9 seconds.
Training sales: 265,283
September sales scored: 11,186


,Model,Train R²,September R²,R² change vs baseline,September MAE,September RMSE,Negative predictions,Fit seconds
0,Linear Regression,0.7767,0.7081,+0.0000,"$308,208","$768,355",103,—
1,Decision Tree,0.6515,0.6093,-0.0988,"$385,945","$888,947",0,3.8
2,Random Forest,0.7579,0.6781,-0.0300,"$332,318","$806,861",0,112.9


In [10]:
print("Training rows:", len(final_train))
print("September rows:", len(september))

Training rows: 265283
September rows: 11186


In [11]:
import pandas as pd

def add_property_features(sales):
    result = sales.copy()

    beds = pd.to_numeric(result["BedroomsTotal"], errors="coerce")
    baths = pd.to_numeric(result["BathroomsTotalInteger"], errors="coerce")
    year_built = pd.to_numeric(result["YearBuilt"], errors="coerce")

    # Recover missing values that the prepared CSV filled with medians.
    for column, values in [
        ("BedroomsTotal", beds),
        ("BathroomsTotalInteger", baths),
        ("YearBuilt", year_built),
    ]:
        flag = result[f"{column}_missing_flag"]
        was_missing = flag.astype(str).str.strip().str.lower().isin(["true", "1"])
        values.loc[was_missing] = float("nan")

    result["bed_bath_ratio"] = beds.div(baths.where(baths > 0))
    sale_year = pd.to_datetime(result["CloseDate"], errors="raise").dt.year
    age = sale_year - year_built
    result["property_age_years"] = age.where(age >= 0)

    return result

train_engineered = add_property_features(final_train)
september_engineered = add_property_features(september)

new_numeric = ["bed_bath_ratio", "property_age_years"]

for name, frame in [
    ("Training", train_engineered),
    ("September test", september_engineered),
]:
    print(f"\n{name}: {len(frame):,} sales")
    print("Missing values in new features:")
    print(frame[new_numeric].isna().sum())
    print("Coordinate columns present:",
          [column for column in ["Latitude", "Longitude"] if column in frame])


Training: 265,283 sales
Missing values in new features:
bed_bath_ratio        146
property_age_years    214
dtype: int64
Coordinate columns present: []

September test: 11,186 sales
Missing values in new features:
bed_bath_ratio        3
property_age_years    9
dtype: int64
Coordinate columns present: []


In [13]:
from pathlib import Path
import zipfile
import re
import numpy as np
import pandas as pd

zip_path = Path("/content/IDX_Analysis 2.zip")

def normalize_key(values):
    return (
        values.astype("string").str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

coordinate_parts = []

with zipfile.ZipFile(zip_path) as archive:
    sold_names = [
        name for name in archive.namelist()
        if "__MACOSX" not in name
        and re.fullmatch(
            r"CRMLSSold\d{6}(?:_filled)?\s*\.csv",
            Path(name).name,
            flags=re.IGNORECASE
        )
    ]

    if len(sold_names) != 33:
        raise ValueError(
            f"Expected 33 monthly sold files; found {len(sold_names)}."
        )

    for name in sorted(sold_names):
        with archive.open(name) as file:
            coordinates = pd.read_csv(
                file,
                usecols=[
                    "ListingKey", "CloseDate",
                    "Latitude", "Longitude"
                ],
                dtype={"ListingKey": "string"},
                low_memory=False
            )

        coordinates["ListingKey"] = normalize_key(
            coordinates["ListingKey"]
        )

        coordinates["sale_date_key"] = pd.to_datetime(
            coordinates["CloseDate"], errors="coerce", utc=True
        ).dt.normalize()

        for column in ["Latitude", "Longitude"]:
            coordinates[column] = pd.to_numeric(
                coordinates[column], errors="coerce"
            ).replace([np.inf, -np.inf], np.nan)

        coordinate_parts.append(
            coordinates.drop(columns="CloseDate")
        )

coordinates = pd.concat(coordinate_parts, ignore_index=True)

join_keys = ["ListingKey", "sale_date_key"]

coordinates = (
    coordinates.dropna(subset=join_keys)
    .drop_duplicates()
)

# Flag conflicting coordinate pairs for the same listing and sale date.
coordinates["coordinate_conflict_flag"] = coordinates.duplicated(
    subset=join_keys, keep=False
)

# Leave conflicting coordinates missing for review.
coordinates.loc[
    coordinates["coordinate_conflict_flag"],
    ["Latitude", "Longitude"]
] = np.nan

coordinates = coordinates.drop_duplicates(subset=join_keys)

def attach_coordinates(data):
    result = data.drop(
        columns=[
            "Latitude", "Longitude", "coordinate_conflict_flag",
            "sale_date_key"
        ],
        errors="ignore"
    ).copy()

    result["ListingKey"] = normalize_key(result["ListingKey"])
    result["sale_date_key"] = pd.to_datetime(
        result["CloseDate"], errors="coerce", utc=True
    ).dt.normalize()

    original_count = len(result)

    result = result.merge(
        coordinates,
        on=join_keys,
        how="left",
        validate="many_to_one",
        sort=False
    )

    assert len(result) == original_count
    return result.drop(columns="sale_date_key")

train_engineered = attach_coordinates(train_engineered)
september_engineered = attach_coordinates(september_engineered)

for name, data in [
    ("Training", train_engineered),
    ("September", september_engineered)
]:
    missing_pair = data[["Latitude", "Longitude"]].isna().any(axis=1)

    print(f"\n{name} rows: {len(data):,}")
    print(f"Rows missing either coordinate: {missing_pair.sum():,}")
    print(
        "Rows with conflicting source coordinates:",
        int(data["coordinate_conflict_flag"].fillna(False).sum())
    )

print("\nCoordinates attached. Next we will check their geographic validity.")


Training rows: 265,283
Rows missing either coordinate: 597
Rows with conflicting source coordinates: 0

September rows: 11,186
Rows missing either coordinate: 83
Rows with conflicting source coordinates: 0

Coordinates attached. Next we will check their geographic validity.


In [14]:
for name, data in [
    ("Training", train_engineered),
    ("September", september_engineered)
]:
    latitude = pd.to_numeric(data["Latitude"], errors="coerce")
    longitude = pd.to_numeric(data["Longitude"], errors="coerce")

    data["coordinate_missing_flag"] = (
        latitude.isna() | longitude.isna()
    )

    data["coordinate_zero_flag"] = (
        latitude.eq(0) | longitude.eq(0)
    )

    # Broad screening bounds around California.
    inside_box = (
        latitude.between(32.0, 42.1)
        & longitude.between(-124.6, -114.0)
    )

    data["coordinate_outside_ca_box_flag"] = (
        ~data["coordinate_missing_flag"]
        & ~data["coordinate_zero_flag"]
        & ~inside_box
    )

    data["coordinate_usable_flag"] = (
        ~data["coordinate_missing_flag"]
        & ~data["coordinate_zero_flag"]
        & inside_box
    )

    print(f"\n{name} sales: {len(data):,}")
    print("Missing coordinates:",
          int(data["coordinate_missing_flag"].sum()))
    print("Zero coordinates:",
          int(data["coordinate_zero_flag"].sum()))
    print("Outside California screening box:",
          int(data["coordinate_outside_ca_box_flag"].sum()))
    print("Ready for district matching:",
          int(data["coordinate_usable_flag"].sum()))


Training sales: 265,283
Missing coordinates: 597
Zero coordinates: 34
Outside California screening box: 41
Ready for district matching: 264611

September sales: 11,186
Missing coordinates: 83
Zero coordinates: 2
Outside California screening box: 1
Ready for district matching: 11100


In [21]:
from pathlib import Path
import zipfile
import tempfile
import geopandas as gpd

district_zip = Path("/content/DistrictAreas2425.zip")

if not district_zip.is_file():
    raise FileNotFoundError(f"Please upload {district_zip.name}")

district_folder = Path(
    tempfile.mkdtemp(prefix="district_boundaries_", dir="/content")
)

with zipfile.ZipFile(district_zip) as archive:
    archive.extractall(district_folder)

shapefiles = [
    path for path in district_folder.rglob("*.shp")
    if "__MACOSX" not in path.parts
    and not path.name.startswith("._")
]

if len(shapefiles) != 1:
    raise ValueError(
        f"Expected one shapefile; found {len(shapefiles)}: "
        f"{[str(path) for path in shapefiles]}"
    )

districts = gpd.read_file(shapefiles[0])

print("District boundary records:", len(districts))
print("Coordinate reference system:", districts.crs)
print("\nAvailable columns:")
print(districts.columns.tolist())

print("\nSample district attributes:")
display(districts.drop(columns="geometry").head())

print("\nMissing geometries:", districts.geometry.isna().sum())
print("Invalid geometries:", (~districts.geometry.is_valid).sum())

District boundary records: 937
Coordinate reference system: EPSG:3857

Available columns:
['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName', 'DistrictNa', 'DistrictTy', 'GradeLow', 'GradeHigh', 'GradeLowCe', 'GradeHighC', 'AssistStat', 'CongressUS', 'SenateCA', 'AssemblyCA', 'UpdateNote', 'EnrollTota', 'EnrollChar', 'EnrollNonC', 'AAcount', 'AApct', 'AIcount', 'AIpct', 'AScount', 'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount', 'PIpct', 'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct', 'ELcount', 'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct', 'MIGcount', 'MIGpct', 'SWDcount', 'SWDpct', 'SEDcount', 'SEDpct', 'DistrctAre', 'Shape__Are', 'Shape__Len', 'geometry']

Sample district attributes:


,OBJECTID,Year,FedID,CDCode,CDSCode,CountyName,DistrictNa,DistrictTy,GradeLow,GradeHigh,...,HOMpct,MIGcount,MIGpct,SWDcount,SWDpct,SEDcount,SEDpct,DistrctAre,Shape__Are,Shape__Len
0,1,2024-25,0601770,0161119,01611190000000,Alameda,Alameda Unified,Unified,PK,12,...,0.7,0,0.0,1356,12.6,3958,36.7,11.248939,4.755489e+07,56522.982683
1,2,2024-25,0601860,0161127,01611270000000,Alameda,Albany City Unified,Unified,PK,12,...,0.6,0,0.0,357,9.7,1184,32.1,1.789984,7.096327e+06,12696.382797
2,3,2024-25,0604740,0161143,01611430000000,Alameda,Berkeley Unified,Unified,PK,12,...,2.1,0,0.0,1111,12.1,2686,29.3,10.434329,4.364648e+07,43695.341538
3,4,2024-25,0607800,0161150,01611500000000,Alameda,Castro Valley Unified,Unified,PK,12,...,1.3,0,0.0,1113,11.6,3728,39.0,66.885571,2.838285e+08,142492.767565
4,5,2024-25,0612630,0161168,01611680000000,Alameda,Emery Unified,Unified,PK,12,...,2.5,0,0.0,81,13.7,412,69.8,1.273929,5.363392e+06,13741.272894



Missing geometries: 0
Invalid geometries: 7


“The district boundaries loaded successfully. Seven geometries need repair.”

In [22]:
# Keep the fields needed for matching properties to districts.
districts_clean = districts[
    ["CDCode", "DistrictNa", "DistrictTy", "geometry"]
].copy()

# Repair invalid boundary shapes.
invalid = ~districts_clean.geometry.is_valid

districts_clean.loc[invalid, "geometry"] = (
    districts_clean.loc[invalid, "geometry"].make_valid()
)

# Use the same coordinate system as property latitude/longitude.
districts_clean = districts_clean.to_crs("EPSG:4326")

districts_clean["CDCode"] = (
    districts_clean["CDCode"].astype("string").str.strip()
)

districts_clean["DistrictTy"] = (
    districts_clean["DistrictTy"].astype("string").str.strip()
)

print("District records:", len(districts_clean))
print("Coordinate system:", districts_clean.crs)
print(
    "Invalid geometries remaining:",
    int((~districts_clean.geometry.is_valid).sum())
)
print("Empty geometries:", int(districts_clean.geometry.is_empty.sum()))

print("\nDistrict types:")
print(districts_clean["DistrictTy"].value_counts(dropna=False))

print("\nGeometry types after repair:")
print(districts_clean.geometry.geom_type.value_counts())

District records: 937
Coordinate system: EPSG:4326
Invalid geometries remaining: 0
Empty geometries: 0

District types:
DistrictTy
Elementary    516
Unified       345
High           76
Name: count, dtype: Int64

Geometry types after repair:
Polygon         889
MultiPolygon     48
Name: count, dtype: int64


“All district boundaries are valid now, and we found three district types.”

In [23]:
import geopandas as gpd
import pandas as pd

def attach_school_districts(data):
    result = data.copy()
    original_rows = len(result)

    if not result.index.is_unique:
        raise ValueError("The sales table needs a unique row index.")

    usable = result["coordinate_usable_flag"].fillna(False)

    points = gpd.GeoDataFrame(
        index=result.index[usable],
        geometry=gpd.points_from_xy(
            result.loc[usable, "Longitude"],
            result.loc[usable, "Latitude"]
        ),
        crs="EPSG:4326"
    )

    for district_type in ["Elementary", "High", "Unified"]:
        column = f"school_district_{district_type.lower()}"
        conflict_column = f"{column}_ambiguous_flag"

        boundaries = districts_clean.loc[
            districts_clean["DistrictTy"].eq(district_type),
            ["CDCode", "geometry"]
        ].copy()

        # Include homes exactly on a boundary, then flag multiple matches.
        joined = gpd.sjoin(
            points,
            boundaries,
            how="left",
            predicate="intersects"
        )

        matches = (
            joined.dropna(subset=["CDCode"])
            .groupby(level=0)["CDCode"]
            .agg(lambda values: sorted(set(values)))
        )

        result[column] = "Unknown"
        result[conflict_column] = False

        single_match = matches[matches.map(len).eq(1)]
        result.loc[single_match.index, column] = single_match.map(
            lambda values: str(values[0])
        )

        multiple_matches = matches[matches.map(len).gt(1)]
        result.loc[multiple_matches.index, column] = "Ambiguous"
        result.loc[multiple_matches.index, conflict_column] = True

    assert len(result) == original_rows
    return result


train_engineered = attach_school_districts(train_engineered)
september_engineered = attach_school_districts(september_engineered)

district_columns = [
    "school_district_elementary",
    "school_district_high",
    "school_district_unified"
]

for name, data in [
    ("Training", train_engineered),
    ("September", september_engineered)
]:
    print(f"\n{name} sales: {len(data):,}")

    summary = pd.DataFrame({
        "Matched": [
            (~data[column].isin(["Unknown", "Ambiguous"])).sum()
            for column in district_columns
        ],
        "Unknown": [
            data[column].eq("Unknown").sum()
            for column in district_columns
        ],
        "Ambiguous": [
            data[column].eq("Ambiguous").sum()
            for column in district_columns
        ]
    }, index=district_columns)

    display(summary)


Training sales: 265,283


,Matched,Unknown,Ambiguous
school_district_elementary,66346,198937,0
school_district_high,64967,200316,0
school_district_unified,199426,65857,0



September sales: 11,186


,Matched,Unknown,Ambiguous
school_district_elementary,2788,8398,0
school_district_high,2740,8446,0
school_district_unified,8353,2833,0


“The district matching is complete, and no sales have ambiguous matches.”

In [24]:
district_columns = [
    "school_district_elementary",
    "school_district_high",
    "school_district_unified"
]

for name, data in [
    ("Training", train_engineered),
    ("September", september_engineered)
]:
    matched_any = (
        ~data[district_columns].isin(["Unknown", "Ambiguous"])
    ).any(axis=1)

    usable = data["coordinate_usable_flag"].fillna(False)

    print(f"\n{name}")
    print(f"Matched at least one district: {matched_any.sum():,}")
    print(f"District coverage: {matched_any.mean():.2%}")
    print(f"No district match: {(~matched_any).sum():,}")
    print(
        "Usable coordinates but no district match:",
        f"{(usable & ~matched_any).sum():,}"
    )


Training
Matched at least one district: 264,580
District coverage: 99.73%
No district match: 703
Usable coordinates but no district match: 31

September
Matched at least one district: 11,100
District coverage: 99.23%
No district match: 86
Usable coordinates but no district match: 0


“School district matching covered 99.73% of training sales and 99.23% of September sales.”

In [25]:
import numpy as np
import pandas as pd
from time import perf_counter

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Recover missing predictors before fitting fresh preprocessing.
train_comparison = train_engineered.copy()
test_comparison = september_engineered.copy()

for data in [train_comparison, test_comparison]:
    for flag in flags:
        data[flag] = (
            data[flag].astype(str).str.strip().str.lower()
            .isin(["true", "1"])
        )

    for column in numeric:
        data[column] = data[column].mask(
            data[f"{column}_missing_flag"]
        )

property_features = ["bed_bath_ratio", "property_age_years"]

feature_sets = {
    "Original": (numeric, categories),
    "Property features": (numeric + property_features, categories),
    "Property + districts": (
        numeric + property_features,
        categories + district_columns
    )
}

model_templates = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=12,
        min_samples_leaf=50,
        random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=50,
        max_depth=16,
        min_samples_leaf=10,
        max_features=0.5,
        random_state=42,
        n_jobs=2
    )
}

feature_results = []
y_training = train_comparison["ClosePrice"]
y_september = test_comparison["ClosePrice"]

for feature_name, (numeric_columns, category_columns) in feature_sets.items():
    preprocessing = ColumnTransformer([
        ("numeric", make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler()
        ), numeric_columns),
        ("categories", OneHotEncoder(
            handle_unknown="ignore"
        ), category_columns),
        ("flags", "passthrough", flags)
    ], sparse_threshold=1.0)

    # Fit preprocessing on training sales only.
    training_matrix = preprocessing.fit_transform(train_comparison)
    september_matrix = preprocessing.transform(test_comparison)

    for model_name, template in model_templates.items():
        print(f"Training {model_name} — {feature_name}", flush=True)

        estimator = clone(template)
        start = perf_counter()
        estimator.fit(training_matrix, y_training)
        fit_seconds = perf_counter() - start

        predicted = estimator.predict(september_matrix)

        feature_results.append({
            "Model": model_name,
            "Feature set": feature_name,
            "Features": training_matrix.shape[1],
            "September R²": r2_score(y_september, predicted),
            "September MAE": mean_absolute_error(y_september, predicted),
            "September RMSE": np.sqrt(
                mean_squared_error(y_september, predicted)
            ),
            "Negative predictions": int((predicted < 0).sum()),
            "Fit seconds": fit_seconds
        })

feature_comparison = pd.DataFrame(feature_results)

display(feature_comparison.style.format({
    "September R²": "{:.4f}",
    "September MAE": "${:,.0f}",
    "September RMSE": "${:,.0f}",
    "Fit seconds": "{:.1f}"
}))

Training Linear Regression — Original
Training Decision Tree — Original
Training Random Forest — Original
Training Linear Regression — Property features
Training Decision Tree — Property features
Training Random Forest — Property features
Training Linear Regression — Property + districts
Training Decision Tree — Property + districts
Training Random Forest — Property + districts


,Model,Feature set,Features,September R²,September MAE,September RMSE,Negative predictions,Fit seconds
0,Linear Regression,Original,4113,0.7081,"$308,208","$768,355",103,12.6
1,Decision Tree,Original,4113,0.6093,"$385,945","$888,947",0,3.8
2,Random Forest,Original,4113,0.6781,"$332,318","$806,861",0,112.0
3,Linear Regression,Property features,4115,0.7143,"$301,920","$760,222",60,11.9
4,Decision Tree,Property features,4115,0.6089,"$386,147","$889,449",0,4.9
5,Random Forest,Property features,4115,0.6776,"$330,881","$807,558",0,140.7
6,Linear Regression,Property + districts,4917,0.7287,"$299,058","$740,808",70,22.0
7,Decision Tree,Property + districts,4917,0.6343,"$371,925","$860,090",0,4.8
8,Random Forest,Property + districts,4917,0.6958,"$315,435","$784,389",0,180.1


“Adding property and school district features improved all three models overall. Linear Regression remained the best performer.”

In [26]:
from pathlib import Path
import shutil
from google.colab import files

output_folder = Path("/content/week6_feature_engineering_output")
output_folder.mkdir(parents=True, exist_ok=True)

feature_comparison.to_csv(
    output_folder / "feature_comparison.csv",
    index=False
)

train_engineered.to_csv(
    output_folder / "train_engineered.csv",
    index=False
)

september_engineered.to_csv(
    output_folder / "september_engineered.csv",
    index=False
)

zip_path = shutil.make_archive(
    str(output_folder),
    "zip",
    str(output_folder)
)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Adding bedroom-to-bathroom ratio, property age, and spatially matched school districts improved all three models compared with their original feature sets. Linear Regression achieved the strongest September performance, with R² increasing from 0.7081 to 0.7287 and MAE decreasing from $308,208 to $299,058. School district features improved all three models beyond the property features alone. The comparison used the same 265,283 training sales and 11,186 September sales, with a $17 million training-price cutoff. Results are exploratory comparisons on a previously evaluated September holdout. The 2024–25 district boundaries were applied as a fixed geographic layer across the sale periods.